# NumPy + Pandas — libraries notebook

**Kaise use karein:** ye cells `numpy` aur `pandas` import karte hain. Ye do libraries app ke **offline Python runtime me bundled** hain (`public/pyodide` me wheels), isliye Notebook Lab me Run karne par asli output milega — internet ki zaroorat nahi.

Local Jupyter me chala rahe ho to pehle `pip install numpy pandas` karna padega. Har library cell ke output yahan khaali chhode gaye hain jaan-boojh kar: hum dummy output likhne ke bajaye app me asli run se dikhate hain.

Sequence: NumPy basics → vectorization → aggregation → Pandas Series/DataFrame → CSV → filtering → groupby → merge → cleaning → numpy se mini ML.


## 1. NumPy arrays — creation, dtype, shape

NumPy array ek fixed-type, contiguous block hai. Python list se fark: memory kam, speed zyada, aur operations **element-wise** hote hain (loop ki zaroorat nahi).


In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4])
b = np.array([[1.0, 2.0], [3.0, 4.0]])
print(a, a.dtype, a.shape, a.ndim)
print(b, b.dtype, b.shape)

print('zeros  :', np.zeros((2, 3)))
print('ones   :', np.ones(4))
print('arange :', np.arange(1, 10, 2))
print('linspace:', np.linspace(0, 1, 5))
print('random :', np.random.default_rng(42).integers(0, 10, size=5))

print('int array + float division:', a / 2)
print('astype int:', b.astype(int))

## 2. Vectorization — loop vs array (same kaam, alag speed)

Pehla cell pure Python hai (ye yahin chalta hai). Doosra NumPy ka hai — output app me dikhega. Dhyaan do ki code kitna chhota ho jata hai.


In [ ]:
import time

data = list(range(200_000))

t0 = time.perf_counter()
squares = [x * x for x in data]
loop_ms = (time.perf_counter() - t0) * 1000

print(f'python loop : {loop_ms:.1f} ms for {len(squares)} items')
print('first 5:', squares[:5])

In [ ]:
import numpy as np
import time

arr = np.arange(200_000)
t0 = time.perf_counter()
np_squares = arr * arr
np_ms = (time.perf_counter() - t0) * 1000
print(f'numpy vector: {np_ms:.1f} ms for {len(np_squares)} items')
print('first 5:', np_squares[:5])

print('broadcasting (2x3 + 3):')
print(np.ones((2, 3)) + np.array([1, 2, 3]))

## 3. Indexing, slicing, boolean masks

NumPy me boolean mask se filtering vectorized hoti hai: `arr[arr > 5]`. Ye Pandas filtering ka base hai.


In [ ]:
import numpy as np

arr = np.array([5, 12, 7, 20, 3, 18])
print(arr[0], arr[-1], arr[1:4], arr[::2])

mask = arr > 10
print('mask    :', mask)
print('filtered:', arr[mask])
print('where   :', np.where(arr > 10, 'high', 'low'))

grid = np.arange(1, 13).reshape(3, 4)
print('grid:\n', grid)
print('column 2:', grid[:, 2])
print('row 1   :', grid[1, :])
print('transpose:\n', grid.T)

## 4. Aggregations aur axis ka matlab

`axis=0` **columns ke neeche** (row-wise reduce), `axis=1` **rows ke across**. Yahi confusion interview me sabse zyada poocha jata hai.


In [ ]:
import numpy as np

marks = np.array([[88, 92, 79], [75, 81, 95], [58, 79, 67]])
print('sum all      :', marks.sum())
print('sum per subject (axis=0):', marks.sum(axis=0))
print('sum per student (axis=1):', marks.sum(axis=1))
print('mean/max/min :', marks.mean(), marks.max(), marks.min())
print('std          :', marks.std())
print('argsort (per student rank):', marks.argsort(axis=1))
print('cumulative sum:', marks.cumsum(axis=1))

scores = np.array([10, 20, 30, 40, 50])
print('normalised (z-score):', ((scores - scores.mean()) / scores.std()))

## 5. Pandas Series — labelled 1-D data

Series = values + index labels. `value_counts`, `map`, `apply` aur boolean filtering iske main tools hain.


In [ ]:
import pandas as pd

marks = pd.Series([88, 92, 75, 81, 95], index=['Ahetesham', 'Sara', 'Ravi', 'Neha', 'Priya'], name='marks')
print(marks)
print('\nvalues:', marks.values, '| dtype:', marks.dtype)
print('describe:\n', marks.describe())
print('label lookup:', marks['Sara'], '| positional:', marks.iloc[1])
print('pass marks (>80):\n', marks[marks > 80])

grades = marks.map(lambda m: 'A' if m >= 90 else ('B' if m >= 75 else 'C'))
print('grades:\n', grades)
print('grade counts:\n', grades.value_counts())

## 6. DataFrame — table banane ke do tarike

Dict-of-lists se DataFrame banao, ya bundled CSV padho. App ke runtime me `students.csv`, `sales.csv`, `customers.csv`, `employees.csv`, `expenses.csv`, `chat_logs.csv` pehle se maujood hain (offline seed ki gayi).


In [ ]:
import pandas as pd

df = pd.DataFrame({
    'student': ['Ahetesham', 'Sara', 'Ravi', 'Neha'],
    'city': ['Pune', 'Delhi', 'Mumbai', 'Pune'],
    'marks': [88, 92, 75, 81],
})
print(df)
print('\nshape:', df.shape, '| columns:', list(df.columns))
print('\ndtypes:\n', df.dtypes)
print('\nhead(2):\n', df.head(2))
print('\ntail(1):\n', df.tail(1))
print('\ndescribe:\n', df.describe())
print('\ninfo:')
df.info()

In [ ]:
import pandas as pd

students = pd.read_csv('students.csv')
sales = pd.read_csv('sales.csv')
print(students)
print('\nsales head:\n', sales.head())
print('\nnulls per column:\n', students.isna().sum())
print('\ncity counts:\n', students['city'].value_counts())
print('\nmarks stats:\n', students['marks'].agg(['count', 'mean', 'median', 'max', 'min']))

## 7. Selection — loc (label) vs iloc (position)

`loc` labels par kaam karta hai (inclusive slicing), `iloc` positions par (exclusive, Python jaisa). Filtering ke baad nayi DataFrame milti hai — original safe rehti hai.


In [ ]:
import pandas as pd

students = pd.read_csv('students.csv').set_index('id')
print('loc row 3:\n', students.loc[3])
print('\niloc first 2 rows, first 2 cols:\n', students.iloc[:2, :2])
print('\ncolumn only:\n', students['name'].head(3))

it_students = students[students['city'] == 'Pune']
print('\nPune students:\n', it_students[['name', 'marks']])
print('\nmarks > 80 and Delhi:\n', students[(students['marks'] > 80) & (students['city'] == "Delhi")])
print('\nwith query():\n', students.query('marks > 85').sort_values('marks', ascending=False))

## 8. groupby — SQL ka GROUP BY, DataFrame me

`groupby(...).agg({...})` me column-wise different functions de sakte ho — interview me yahi pattern maanga jata hai.


In [ ]:
import pandas as pd

students = pd.read_csv('students.csv')
print('per city:\n', students.groupby('city')['marks'].agg(['count', 'mean', 'max']).round(1))
print('\ncity wise top scorer:\n', students.loc[students.groupby('city')['marks'].idxmax()])

sales = pd.read_csv('sales.csv')
by_product = sales.groupby('product').agg(orders=('order_id', 'nunique'), units=('qty', 'sum'), revenue=('revenue', 'sum'))
by_product['revenue_share_%'] = (by_product['revenue'] / by_product['revenue'].sum() * 100).round(1)
print('\nproduct performance:\n', by_product.sort_values('revenue', ascending=False))
print('\ntop product:', by_product['revenue'].idxmax())

## 9. Merge/join — students + customers

`merge` SQL join hai: `how='inner' | 'left' | 'right' | 'outer'`, aur `on=` common column. Join ke baad row count check karna habit banao (duplicate keys se rows badh jati hain).


In [ ]:
import pandas as pd

students = pd.read_csv('students.csv')
customers = pd.read_csv('customers.csv')
print('customers:\n', customers.head())

joined = students.merge(customers, on='id', how='left', suffixes=('_student', '_customer'))
print('\njoined rows:', len(joined), 'vs students:', len(students))
print(joined.head())

print('\nmean marks by joined city:\n', joined.groupby('city_student')['marks'].mean().round(1))
print('\nmissing city after join:\n', joined['city_customer'].isna().sum())

## 10. Cleaning — missing values, types, string accessor

Real data me null, galat dtype aur extra spaces hote hain. `.str` accessor se text columns par vectorized string methods chalti hain.


In [ ]:
import numpy as np
import pandas as pd

raw = pd.DataFrame({
    'Name': [' Neha ', 'rahul', 'PRIYA', None],
    'City': ['pune', 'delhi', 'Mumbai', np.nan],
    'Marks': ['88', '75', '91', '80'],
})
print('raw:\n', raw)
print('nulls:\n', raw.isna().sum())

clean = raw.copy()
clean['Name'] = clean['Name'].str.strip().str.title()
clean['City'] = clean['City'].str.title().fillna('Unknown')
clean['Marks'] = pd.to_numeric(clean['Marks'], errors='coerce')
print('\ncleaned:\n', clean)
print('\ndtypes:\n', clean.dtypes)

clean = clean.dropna(subset=['Name'])
clean['passed'] = clean['Marks'] >= 80
print('\nfinal:\n', clean)
print('\npass rate: {:.0%}'.format(clean['passed'].mean()))

## 11. NumPy se chhota ML — linear regression

Ye classic interview demo hai: scatter data par best-fit line, sab kuch NumPy se (scikit-learn ki zaroorat nahi).


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
x = np.arange(1, 21)
y = 3 * x + 5 + rng.normal(0, 3, size=x.size)

# closed-form least squares: theta = (X^T X)^-1 X^T y
X = np.column_stack([np.ones_like(x), x])
theta = np.linalg.inv(X.T @ X) @ X.T @ y
print('intercept, slope:', theta.round(2))

predictions = X @ theta
mse = np.mean((y - predictions) ** 2)
r2 = 1 - mse / np.var(y)
print('mse:', round(mse, 2), '| r2:', round(r2, 3))

print('polyfit check  :', np.polyfit(x, y, 1).round(2))
print('prediction x=25:', round(float(np.array([1, 25]) @ theta), 2))

## 12. Export — CSV/JSON nikalna

Analysis ka aakhri step: result ko file me likho (`to_csv`) ya API/LLM ke liye JSON banao (`to_json`). App me ye virtual filesystem me likhta hai.


In [ ]:
import pandas as pd

sales = pd.read_csv('sales.csv')
summary = (sales.groupby('product')['revenue'].sum().sort_values(ascending=False)
           .reset_index().rename(columns={'revenue': 'total_revenue'}))
print(summary)

summary.to_csv('product_summary.csv', index=False)
print('\nwritten product_summary.csv')
print('round trip:\n', pd.read_csv('product_summary.csv').head())

print('\nJSON export:')
print(summary.head(3).to_json(orient='records', indent=2))

## 13. Debugging — ye cell app me error dega (jaan-boojh kar)

Error: `KeyError` ya `ValueError` — column ka naam galat. App me Run karke asli traceback padho, phir fix wala cell chalao.


In [ ]:
import pandas as pd

students = pd.read_csv('students.csv')
print(students.groupby('City')['marks'].mean())

In [ ]:
import pandas as pd

students = pd.read_csv('students.csv')
print('columns:', list(students.columns))          # pehle columns dekho, guess na karo
print(students.groupby('city')['marks'].mean().round(1))

## 14. Practice set

1. `expenses.csv` se total spend nikaalo aur daily average do.
2. `employees.csv` me department-wise max salary aur us employee ka naam.
3. `sales.csv` me per product units aur revenue ka bar-chart-ready DataFrame banao.
4. `students.csv` me city-wise pass rate (>80) nikalo.
5. NumPy se ek 5x5 matrix banao, uska determinant aur inverse nikaalo.
6. Do arrays ka element-wise cosine similarity nikaalo (dot / magnitude product).
7. Ek column ko 0-1 me min-max normalize karo (`(x - min) / (max - min)`).
8. `chat_logs.csv` me message length ka histogram-ready data banao (value_counts on length buckets).


In [ ]:
import numpy as np
import pandas as pd

# 1
expenses = pd.read_csv('expenses.csv')
print('expenses columns:', list(expenses.columns), '| rows:', len(expenses))
print(expenses.describe())

# 2
employees = pd.read_csv('employees.csv')
print('\nemployees:\n', employees.head())
top = employees.loc[employees.groupby('department')['salary'].idxmax()]
print('\ndepartment top earners:\n', top[['name', 'department', 'salary']])

# 3
sales = pd.read_csv('sales.csv')
per_product = sales.groupby('product').agg(units=('qty', 'sum'), revenue=('revenue', 'sum')).reset_index()
print('\nper product:\n', per_product.sort_values('revenue', ascending=False))

# 4
students = pd.read_csv('students.csv')
students['passed'] = students['marks'] > 80
print('\npass rate by city:\n', students.groupby('city')['passed'].mean().round(2))

# 5
matrix = np.arange(1, 26).reshape(5, 5)
print('\ndet:', round(float(np.linalg.det(matrix)), 3))
print('inverse (first row):', np.linalg.inv(matrix)[0].round(3))
print('rank:', np.linalg.matrix_rank(matrix))

# 6
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 3.0, 4.0])
cosine = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
print('\ncosine similarity:', round(cosine, 3))

# 7
marks = students['marks'].to_numpy()
normalised = (marks - marks.min()) / (marks.max() - marks.min())
print('\nnormalised marks:', normalised.round(2))

# 8
logs = pd.read_csv('chat_logs.csv')
print('\nlog columns:', list(logs.columns))
lengths = logs.iloc[:, -1].astype(str).str.len()
print('length buckets:\n', pd.cut(lengths, bins=[0, 20, 40, 80, 200]).value_counts().sort_index())

## 15. Interview drills

1. NumPy array Python list se tez kyun hota hai? (dtype, contiguous memory, C loops, no boxing)
2. `axis=0` aur `axis=1` ka exact matlab batao aggregation me.
3. `loc` vs `iloc` — ek example se difference.
4. `SettingWithCopyWarning` kya hai aur `df.copy()` se kaise bachate ho?
5. `apply` slow kyun hota hai? Vectorized alternative kya hai?
6. 10 GB CSV Pandas me kaise padhoge? (`chunksize`, `usecols`, `dtype`, ya Polars/DuckDB)


In [ ]:
import numpy as np
import pandas as pd
import time

df = pd.DataFrame({'x': np.arange(500_000), 'y': np.arange(500_000)})

t0 = time.perf_counter()
df['z_apply'] = df['x'].apply(lambda v: v * 2)
apply_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
df['z_vector'] = df['x'] * 2
vector_ms = (time.perf_counter() - t0) * 1000

print(f'apply : {apply_ms:.0f} ms')
print(f'vector: {vector_ms:.0f} ms')
print('same result:', bool((df['z_apply'] == df['z_vector']).all()))
print('memory usage (MB):', round(df.memory_usage(deep=True).sum() / 1e6, 1))